In [1]:
from pathlib import Path
import pandas as pd 
import numpy as np 
import openmatrix as omx

In [2]:
output_root= "C:/Users/USJC173858/temp"
scenarios = {
    "BICOUNTY_MODEL": "../data/external/ccta/nonres",
    "TM-1.6 SW data": "C:/Users/USJC173858/WSP O365/MTC_TM1.7 - Documents/travel-model-one/utilities/trucks/trucks_2026_update/data/interim/matrix_projection/sw_od_trips_with_mtc_format", 
    "TM-1.6": 'C:/temp/mtc_cube_runs/TM-1.6/nonres',
    "TM-1.7_GEN_REEST": 'C:/temp/mtc_cube_runs/TM-1.7_GEN_REEST/nonres',
    "TM-1.7_GEN_NEWSPEC": 'C:/temp/mtc_cube_runs/TM-1.7_GEN_NEWSPEC/nonres',
    "TM-1.7_GEN_IX": 'C:/temp/mtc_cube_runs/TM-1.7_GEN_IX/nonres',
    "TM-1.6_FIX_ROUNDING_ISSUE": "C:/temp/mtc_cube_runs/TM-1.6_FIX_ROUNDING_ISSUE/nonres",
}
path_pattern = "TripsTrk{tod}x.omx"
tods = ["EA", "AM", "MD", "PM", "EV"]

In [3]:
rows = []
for scenario, scenario_path in scenarios.items(): 
    for tod in tods: 
        fname = path_pattern.format(tod=tod)
        ref_path = Path(scenario_path, fname)
        reference_omx = omx.open_file(ref_path, "r")
        for truck_type in reference_omx.list_matrices():
            m = np.array(reference_omx[truck_type])
            internal_break = 1454
            correction_factor = 1
            
            if scenario == "BICOUNTY_MODEL":
                # This is the last TAZ in the 9-county Bay Area Region. 
                # 6274-6594 Are San Joaquin Country trips. 
                # Remanining are model gateways. 
                internal_break = 6272 
                correction_factor = 0 # Avoid county  SanJoaquin-to-SanJoaquin trips. 
                m = m/100 # BICOUNTY_MODEL outputs are multiplied by 100 to avoid rounding issues. 
  
            ii = m[:internal_break][:,:internal_break].sum()
            ix = m[:internal_break][:,internal_break:].sum()
            xi = m[internal_break:][:,:internal_break].sum()
            xx = m[internal_break:][:,internal_break:].sum() * correction_factor
            # print(f"{scenario}_{truck_type}_{tod}: {m.sum():.0f}")
        
            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "internal-internal",
                "truck_trips": ii, 
            })

            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "internal-external",
                "truck_trips": ix, 
            })

            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "external-internal",
                "truck_trips": xi, 
            })

            rows.append({
                "scenario": scenario,
                "tod": tod,
                "truck_type": truck_type,
                "type": "external-external",
                "truck_trips": xx, 
            })
        reference_omx.close()
df = pd.DataFrame(rows)

In [6]:
all_scenarios = ["TM-1.6", "TM-1.6_FIX_ROUNDING_ISSUE","TM-1.6 SW data", "TM-1.7_GEN_REEST", "TM-1.7_GEN_NEWSPEC", "TM-1.7_GEN_IX"]
data_scenarios = ["BICOUNTY_MODEL", "TM-1.6", "TM-1.6_FIX_ROUNDING_ISSUE","TM-1.6 SW data"]

print ("TABLE 1. Trip Generation by flow type")
df.pivot_table(
    index = ["truck_type", "type"], 
    columns = ["scenario"], 
    values = "truck_trips", 
    aggfunc = "sum"
)[data_scenarios].style.format("{:,.0f}")

TABLE 1. Trip Generation by flow type


In [7]:
print ("TABLE 2. Trip Generation by time of day")
df[df["type"] == "internal-internal"].pivot_table(
    index = ["truck_type","scenario" ], 
    columns = ["tod"], 
    values = "truck_trips", 
    aggfunc = "sum"
)[["EA", "AM","MD","PM", "EV"]].style.format("{:,.0f}")

TABLE 2. Trip Generation by time of day
